# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 colorectal cancer dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the Croissant metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset URL for the FAIR^2 colorectal cancer package
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata via the .metadata attribute
# Display dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs. Reference all entities by their `@id` attribute.

Croissant datasets define groups of records in `recordSet` objects. We'll retrieve all `recordSet` entities and display their fields and key metadata.


In [ ]:
# Show all record sets by their @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set['@id']}")
        name = record_set.get('name', 'N/A')
        fields = record_set.get('field', [])
        print(f"  Name: {name}")
        print(f"  Fields:")
        for field in fields:
            field_id = field.get('@id', None)
            field_name = field.get('name', None)
            print(f"    - {field_id} ({field_name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the Croissant `@id` for the record set and fields.

If there are multiple record sets, we'll load each by its `@id` and preview their columns.

In [ ]:
# If no record sets, abort this section
record_set_ids = []
if dataset.metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
else:
    print("No record sets present in metadata, cannot extract records.")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns and first rows for each record set
for record_set_id, df in dataframes.items():
    print(f"=== RecordSet {record_set_id} DataFrame ===")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalizing numeric fields, grouping by key attributes. All fields are referenced by their `@id`.

We'll select a numeric field for filtering and normalization, then group the records by a chosen categorical field.


In [ ]:
# Example: Choose a record set and numeric field by their @id
# You may change these IDs based on data overview step
if record_set_ids:
    # For demo, select the first record set
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Select field @ids for numeric and group fields by exploring the df columns
    # Choose first numeric-looking field by checking dtype
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Choose the first object-dtype column as group field
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break

    if numeric_field_id is not None:
        # Filter: records above a threshold (10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field for filtered records
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field_id if present
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical group field found in columns.")
    else:
        print("No numeric field found in this record set.")
else:
    print("No record sets present, EDA cannot proceed.")

## 5. Visualization

Visualize distributions or relationships using matplotlib and seaborn. For example, histogram of numeric field or boxplot grouped by a category (all by @id).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Visualize only if EDA found numeric/group fields
if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet {record_set_id})")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot grouped by group_field_id if possible
    if group_field_id is not None:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load FAIR^2 dataset metadata and records using `mlcroissant`.
- Review and extract record sets and fields by `@id`.
- Process and filter data dynamically using field IDs.
- Visualize distributions and relationships between fields.

Refer to the dataset Croissant schema and original publication for further details on the field meanings and clinical interpretations.